# HW3 · 05 — Explainability (bonus): what does the model rely on?
1. **How common are suspected "shortcut" artifacts per class?** (curly quotes, `missing section` placeholders,
   chat phrases like "let me know if", `urllink`) — these showed up as top TF-IDF features in notebook 02.
2. **Token attributions for XLM-R** with Integrated Gradients (Sundararajan et al., 2017), implemented by hand on the
   input embeddings: per-example highlights + words that push towards each class on average.
3. **Perturbation test:** remove/normalise those artifacts in val texts and measure how much macro-F1 drops.
   If the score drops a lot, the model relies on artifacts rather than on writing style.

Runs on Colab (T4) next to the model saved by notebook 03 in Google Drive, or on Kaggle with the 03 output attached.

In [ ]:
import glob, os, re, html
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from IPython.display import HTML, display

SEED = 42
IN_COLAB = not os.path.exists("/kaggle/input")  # Kaggle also has a /content folder, so check for /kaggle/input instead
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/hw3_outputs"
    if not os.path.exists("/content/data/train.csv"):
        files.upload()  # kaggle.json
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.replace("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
        os.system("kaggle competitions download -c ai-generated-text-detection-hw3-ucu-ai-2026 -p /content/data")
        os.system("cd /content/data && unzip -oq *.zip")
    DATA_DIR, MODEL_DIR = "/content/data", f"{OUT}/xlmr_model"
    os.chdir(OUT)
else:
    DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
    MODEL_DIR = os.path.dirname(glob.glob("/kaggle/input/**/xlmr_model/config.json", recursive=True)[0])
VAL_PROBA = glob.glob(("/content/drive/MyDrive/hw3_outputs" if IN_COLAB else "/kaggle/input") + "/**/xlmr_val_proba_headtail.csv", recursive=True)[0]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DATA_DIR, MODEL_DIR, DEVICE)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
TEXT_COL, LABEL_COL = "text", "label"
LANG_COL = next((c for c in ["language", "lang"] if c in train.columns), None)

def make_split(train, label_col="label", lang_col=None, n_val=10_000, n_train=40_000, seed=42):
    strat = train[label_col].astype(str) + ("_" + train[lang_col].astype(str) if lang_col else "")
    rest, val = train_test_split(train, test_size=n_val, stratify=strat, random_state=seed)
    strat_rest = strat.loc[rest.index]
    train_sub, _ = train_test_split(rest, train_size=n_train, stratify=strat_rest, random_state=seed)
    return train_sub.reset_index(drop=True), val.reset_index(drop=True)

train_sub, val = make_split(train, LABEL_COL, LANG_COL)
CLASSES = sorted(train[LABEL_COL].unique())
y_va = val[LABEL_COL].map({c: i for i, c in enumerate(CLASSES)}).values
print("first val ids:", val["id"].head(3).tolist())

## 1. How common are the suspected artifacts in each class? (train_sub only)

In [ ]:
ARTIFACTS = {
    "curly apostrophe/quotes ’ “ ”": r"[’‘“”]",
    "'missing ...' placeholder": r"missing (?:section|part|text|content|paragraph)",
    "chat phrase 'let me know'": r"let me know",
    "'urllink' (blog corpus)": r"urllink",
    "space before punctuation ' .'": r" [.,!?;:](?:\s|$)",
}
freq = pd.DataFrame({name: train_sub.groupby(LABEL_COL)[TEXT_COL].apply(lambda s: s.str.contains(pat, case=False, regex=True).mean())
                     for name, pat in ARTIFACTS.items()}).T.round(3)
display(freq)  # share of texts in each class that contain the pattern

for pat in [r".{0,60}missing (?:section|part|text|content|paragraph).{0,60}", r".{0,60}let me know.{0,40}"]:
    print("\nexamples:", pat)
    for lab in CLASSES:
        m = train_sub[train_sub[TEXT_COL].str.contains(pat, case=False, regex=True)]
        m = m[m[LABEL_COL] == lab].head(2)
        for t in m[TEXT_COL]:
            print(f"  [{lab}]", re.search(pat, t, flags=re.I).group(0).replace("\n", " "))

## 2. Integrated Gradients for XLM-R

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()
emb_layer = model.get_input_embeddings()
SENTENCEPIECE = any(p.startswith("▁") for p in tok.tokenize("hello world"))
MAX_TOK = 256  # shorter than 512 to keep IG fast

def encode(text):
    ids = tok(text, add_special_tokens=False, truncation=False)["input_ids"][:MAX_TOK - 2]
    return [tok.cls_token_id] + ids + [tok.sep_token_id]

def integrated_gradients(ids, target, steps=24):
    """Attribution of each token to the logit of `target`; baseline = all <pad> embeddings."""
    x_ids = torch.tensor([ids], device=DEVICE)
    emb = emb_layer(x_ids).detach()
    base = emb_layer(torch.full_like(x_ids, tok.pad_token_id)).detach()
    alphas = torch.linspace(1 / steps, 1, steps, device=DEVICE).view(-1, 1, 1)
    path = (base + alphas * (emb - base)).requires_grad_(True)        # (steps, L, d): all steps in one batch
    logits = model(inputs_embeds=path, attention_mask=torch.ones(path.shape[:2], device=DEVICE)).logits
    grads, = torch.autograd.grad(logits[:, target].sum(), path)
    attr = ((emb - base)[0] * grads.mean(0)).sum(-1)                   # (L,)
    return attr.detach().cpu().numpy()

def to_words(ids, attr):
    """Merge sub-word pieces into words and sum their attributions (skip <s> and </s>)."""
    pieces = tok.convert_ids_to_tokens(ids)[1:-1]; attr = attr[1:-1]
    words, scores = [], []
    for p, a in zip(pieces, attr):
        new = p.startswith("▁") if SENTENCEPIECE else not p.startswith("##")
        piece = p.lstrip("▁") if SENTENCEPIECE else p.replace("##", "")
        if new or not words:
            words.append(piece); scores.append(a)
        else:
            words[-1] += piece; scores[-1] += a
    return words, np.array(scores)

def predict_proba(text):
    with torch.no_grad():
        return torch.softmax(model(torch.tensor([encode(text)], device=DEVICE)).logits, -1)[0].cpu().numpy()

def highlight(text, target):
    ids = encode(text)
    words, s = to_words(ids, integrated_gradients(ids, target))
    s = s / (np.abs(s).max() + 1e-9)
    spans = [f'<span style="background: rgba({"220,40,40" if v > 0 else "40,90,220"},{abs(v):.2f})">{html.escape(w)}</span>'
             for w, v in zip(words, s)]
    return " ".join(spans)

### Per-example highlights (red = pushes towards the explained class, blue = against)
Two correct and two wrong predictions for each class, explained w.r.t. the *predicted* class.

In [ ]:
p_val = pd.read_csv(VAL_PROBA).set_index("id").loc[val["id"], CLASSES].values
pred_va = p_val.argmax(1)
rng = np.random.default_rng(SEED)
html_parts = []
for ci, c in enumerate(CLASSES):
    for kind, mask in [("correct", (y_va == ci) & (pred_va == ci)), ("wrong", (y_va == ci) & (pred_va != ci))]:
        for i in rng.choice(np.where(mask)[0], size=min(2, mask.sum()), replace=False):
            t = val[TEXT_COL].iloc[i]
            html_parts.append(f"<p><b>true={c} · predicted={CLASSES[pred_va[i]]} ({kind}) · domain={val['domain'].iloc[i]}</b><br>"
                              f"{highlight(t, pred_va[i])}</p><hr>")
page = "\n".join(html_parts)
open("xai_examples.html", "w").write(f"<html><body style='font-family:sans-serif;max-width:900px'>{page}</body></html>")
display(HTML(page))

### Which words push towards each class on average?
IG for the true class on 150 correctly classified val texts per class; average word attribution
(words must occur at least 8 times).

In [ ]:
N_PER_CLASS, MIN_COUNT = 150, 8
top_words = {}
for ci, c in enumerate(CLASSES):
    ok = np.where((y_va == ci) & (pred_va == ci))[0]
    idx = rng.choice(ok, size=min(N_PER_CLASS, len(ok)), replace=False)
    total, count = defaultdict(float), defaultdict(int)
    for i in idx:
        ids = encode(val[TEXT_COL].iloc[i])
        words, s = to_words(ids, integrated_gradients(ids, ci))
        for w, v in zip(words, s):
            total[w] += v; count[w] += 1
    mean = pd.Series({w: total[w] / count[w] for w in total if count[w] >= MIN_COUNT}).sort_values(ascending=False)
    top_words[c] = mean.head(15)
    print(f"{c:7s}:", " | ".join(f"{w!r} {v:.3f}" for w, v in mean.head(15).items()))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, c in zip(axes, CLASSES):
    s = top_words[c][::-1]
    ax.barh([repr(w) for w in s.index], s.values, color="tab:red"); ax.set_title(f"words pushing towards '{c}'")
plt.tight_layout(); plt.savefig("xai_top_words.png", dpi=120); plt.show()

## 3. Perturbation test: does the model need the artifacts?
We change only the texts that contain an artifact, re-predict them with the same head+tail input as in 03,
and compare macro-F1 on the whole val split. Everything else stays identical.

In [ ]:
HEAD, BODY = 256, 510
def head_tail_ids(text):
    x = tok(text[:15000] + " " + text[-15000:] if len(text) > 30000 else text, add_special_tokens=False)["input_ids"]
    x = x if len(x) <= BODY else x[:HEAD] + x[-(BODY - HEAD):]
    return [tok.cls_token_id] + x + [tok.sep_token_id]

def predict_texts(texts, bs=32):
    out = []
    for k in range(0, len(texts), bs):
        seqs = [head_tail_ids(t) for t in texts[k:k + bs]]
        L = max(map(len, seqs))
        ids = torch.tensor([s + [tok.pad_token_id] * (L - len(s)) for s in seqs], device=DEVICE)
        att = (ids != tok.pad_token_id).long()
        with torch.no_grad(), torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            out.append(torch.softmax(model(ids, attention_mask=att).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

PERTURB = {
    "straight quotes (’‘“” → ' \")": lambda t: t.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"'),
    "remove 'missing …' placeholders": lambda t: re.sub(r"[\[\(]?[^.\n\[\(]{0,30}missing (?:section|part|text|content|paragraph)[^\]\)\n.]{0,60}[\]\)]?", " ", t, flags=re.I),
    "remove sentences with 'let me know'": lambda t: re.sub(r"[^.!?\n]*let me know[^.!?\n]*[.!?]?", " ", t, flags=re.I),
    "remove space before punctuation": lambda t: re.sub(r" ([.,!?;:])", r"\1", t),
}
base_f1 = f1_score(y_va, pred_va, average="macro")
rows = [{"perturbation": "none (reference)", "texts changed": 0, "macro_f1": base_f1, "delta": 0.0, "predictions flipped": 0}]
texts = val[TEXT_COL].tolist()
for name, fn in PERTURB.items():
    new_texts = [fn(t) for t in texts]
    changed = np.array([a != b for a, b in zip(texts, new_texts)])
    p = p_val.copy()
    if changed.any():
        p[changed] = predict_texts([t for t, c in zip(new_texts, changed) if c])
    pred = p.argmax(1)
    rows.append({"perturbation": name, "texts changed": int(changed.sum()),
                 "changed per class": {c: int(changed[y_va == i].sum()) for i, c in enumerate(CLASSES)},
                 "macro_f1": f1_score(y_va, pred, average="macro"),
                 "delta": f1_score(y_va, pred, average="macro") - base_f1,
                 "predictions flipped": int((pred != pred_va).sum())})
    print(rows[-1])
perturb = pd.DataFrame(rows); perturb.to_csv("xai_perturbation.csv", index=False)
perturb.round(4)